# E-Commerce Sales & Customer Analytics
### Data Cleaning, Validation, Exploratory Data Analysis & Business KPI Report

This notebook covers the complete analysis pipeline:
1. Load & validate the cleaned e-commerce dataset
2. Data quality checks (missing values, duplicates, invalid ranges, dtypes)
3. Exploratory Data Analysis (EDA) with visualizations
4. Business KPI calculation (Revenue, AOV, ARPU, Discount Leakage, etc.)
5. Segment, category, and geographic analysis
6. Final business recommendations

**Dataset:** `cleaned_ecommerce_data.xlsx` (sheet: `clean_ecommerce_data`)
**Raw source:** `Ecommerce_Raw_Source_Data.xlsx` (Customers, Orders, Payments, Products)


## 1. Setup & Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")

# Load the cleaned dataset (place this file in the same folder as this notebook)
df = pd.read_excel("cleaned_ecommerce_data.xlsx", sheet_name="clean_ecommerce_data")
df.head()

## 2. Data Cleaning & Validation

Quality checks on the cleaned dataset — shape, missing values, duplicates, data types, and range validation for key numeric/date fields.

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
print(df.columns.tolist())

In [ ]:
df.isnull().sum()

In [ ]:
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_pct

In [ ]:
df.duplicated().sum()

In [ ]:
df["OrderID"].duplicated().sum()

In [ ]:
print("Unique Customers:", df["CustomerID"].nunique())
print("Total Rows:", len(df))

In [ ]:
df["CustomerID"].value_counts().head(10)

In [ ]:
df.dtypes

### 2.1 Enforce correct data types

In [ ]:
# ID / count / age columns
df['OrderID'] = pd.to_numeric(df['OrderID'], errors='coerce').astype('Int64')
df['CustomerID'] = pd.to_numeric(df['CustomerID'], errors='coerce').astype('Int64')
df['ProductID'] = pd.to_numeric(df['ProductID'], errors='coerce').astype('Int64')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').astype('Int64')
df['Age'] = pd.to_numeric(df['Age'], errors='coerce').astype('Int64')

# Date columns
df['OrderDate'] = pd.to_datetime(df['OrderDate'], errors='coerce')
df['SignupDate'] = pd.to_datetime(df['SignupDate'], errors='coerce')

# Numeric / financial columns
df['Discount'] = pd.to_numeric(df['Discount'], errors='coerce')
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce')
df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce')
df['OrderValue'] = pd.to_numeric(df['OrderValue'], errors='coerce')

# Text columns
text_columns = [
    'PaymentMethod',
    'Status',
    'City',
    'CustomerSegment',
    'ProductName',
    'Category'
]

for col in text_columns:
    df[col] = df[col].astype('string')

# Final dtype check
print(df.dtypes)

### 2.2 Range & validity checks

In [ ]:
print("Missing Age:", df['Age'].isna().sum())
print("Minimum Age:", df['Age'].min())
print("Maximum Age:", df['Age'].max())
print("Unique Ages:", df['Age'].nunique())

In [ ]:
invalid_age = df[(df['Age'] < 18) | (df['Age'] > 100)]

print("Invalid Age Records:", len(invalid_age))
display(invalid_age)

In [ ]:
print(df['Age'].describe())

In [ ]:
print("Invalid dates:", df["OrderDate"].isnull().sum())

In [ ]:
print("Minimum Date:", df["OrderDate"].min())
print("Maximum Date:", df["OrderDate"].max())

In [ ]:
print("Quantity <= 0:", (df["Quantity"] <= 0).sum())

In [ ]:
print("Unit Price <= 0:", (df["UnitPrice"] <= 0).sum())

In [ ]:
print("Total Sales < 0:", (df["Sales"] < 0).sum())

In [ ]:
print("Discount < 0:", (df["Discount"] < 0).sum())
print("Discount > 100:", (df["Discount"] > 100).sum())

In [ ]:
df["CustomerSegment"].value_counts()

In [ ]:
df["Category"].value_counts()

In [ ]:
df["PaymentMethod"].value_counts()

In [ ]:
for col in df.select_dtypes(include="object").columns:
    spaces = df[col].astype(str).str.strip().ne(df[col].astype(str)).sum()
    print(col, ":", spaces)

In [ ]:
print("City:", df["City"].nunique())
print("Categories:", df["Category"].nunique())
print("Payment Methods:", df["PaymentMethod"].nunique())

In [ ]:
print(df["City"].unique())
print(df["Category"].unique())
print(df["PaymentMethod"].unique())

In [ ]:
df.info()

## 3. Exploratory Data Analysis (EDA)

Statistical summary and visual exploration of sales, products, cities, customers, and payment behavior.

In [ ]:
print("Dataset Shape (Rows, Columns):", df.shape)
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())

In [ ]:
df.describe()

### 3.1 Sales by Category

In [ ]:
category_sales = df.groupby("Category")["Sales"].sum().reset_index()
category_sales = category_sales.sort_values(by="Sales", ascending=False)

print("Total Sales by Category:")
print(category_sales)

plt.figure(figsize=(8, 4))
sns.barplot(data=category_sales, x="Category", y="Sales", palette="viridis")
plt.title("Total Sales by Product Category")
plt.xlabel("Category")
plt.ylabel("Total Sales (₹)")
plt.tight_layout()
plt.show()

### 3.2 Top 5 Products

In [ ]:
top_products = df.groupby("ProductName")["Sales"].sum().reset_index()
top_products = top_products.sort_values(by="Sales", ascending=False).head(5)

plt.figure(figsize=(8, 4))
sns.barplot(data=top_products, x="Sales", y="ProductName", palette="Blues_r")
plt.title("Top 5 Products by Sales")
plt.xlabel("Total Sales")
plt.ylabel("Product Name")
plt.tight_layout()
plt.show()

### 3.3 Sales by City & Customer Age Distribution

In [ ]:
city_sales = df.groupby("City")["Sales"].sum().reset_index().sort_values(by="Sales", ascending=False)

plt.figure(figsize=(10, 4))
sns.barplot(data=city_sales, x="City", y="Sales", palette="magma")
plt.title("Sales Distribution Across Cities")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(df["Age"], bins=15, kde=True, color="skyblue")
plt.title("Customer Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

### 3.4 Payment Method Distribution

In [ ]:
payment_counts = df["PaymentMethod"].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(payment_counts, labels=payment_counts.index, autopct='%1.1f%%', startangle=140)
plt.title("Payment Method Distribution")
plt.tight_layout()
plt.show()

### 3.5 Monthly Sales Trend

In [ ]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"])
monthly_sales = df.resample('ME', on='OrderDate')["Sales"].sum().reset_index()

plt.figure(figsize=(10, 4))
sns.lineplot(data=monthly_sales, x="OrderDate", y="Sales", marker="o", color="green")
plt.title("Monthly Sales Trend")
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.show()

### 3.6 Returns & Net Sales

In [ ]:
returns_df = df[df['Quantity'] < 0]
clean_sales_df = df[df['Quantity'] > 0].copy()

print(f"Total Negative/Returned Transactions: {len(returns_df)}")
print(f"Total Net Sales: ₹{clean_sales_df['Sales'].sum():,.2f}")

### 3.7 Customer Segment Analysis

In [ ]:
segment_analysis = clean_sales_df.groupby('CustomerSegment').agg(
    Total_Sales=('Sales', 'sum'),
    Avg_Order_Value=('Sales', 'mean'),
    Order_Count=('OrderID', 'count')
).reset_index()

print(segment_analysis)

### 3.8 Order Status Breakdown

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='Status', palette='Set2')
plt.title('Order Status Breakdown')
plt.ylabel('Number of Orders')
plt.tight_layout()
plt.show()

### 3.9 Discount Impact on Quantity

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=clean_sales_df, x='Discount', y='Quantity', palette='Blues')
plt.title('Impact of Discount on Quantity Sold')
plt.tight_layout()
plt.show()

### 3.10 Signup-to-First-Order Gap

In [ ]:
df['Days_To_First_Order'] = (df['OrderDate'] - df['SignupDate']).dt.days

plt.figure(figsize=(8, 4))
sns.histplot(df['Days_To_First_Order'], bins=20, kde=True, color='purple')
plt.title('Gap Between Signup and Order Date (in Days)')
plt.xlabel('Days')
plt.ylabel('Number of Orders')
plt.tight_layout()
plt.show()

### 3.11 Discount Revenue Leakage by Category

In [ ]:
clean_sales_df['Potential_Sales'] = clean_sales_df['Quantity'] * clean_sales_df['UnitPrice']
clean_sales_df['Discount_Amount'] = clean_sales_df['Potential_Sales'] - clean_sales_df['Sales']

total_discount_given = clean_sales_df['Discount_Amount'].sum()
print(f"Total Revenue Given Away in Discounts: ₹{total_discount_given:,.2f}")

plt.figure(figsize=(8, 4))
sns.barplot(data=clean_sales_df, x='Category', y='Discount_Amount', estimator=sum, errorbar=None, palette='Reds')
plt.title('Total Discount Revenue Loss by Category')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 3.12 City vs Category Revenue Heatmap

In [ ]:
city_category = pd.crosstab(clean_sales_df['City'], clean_sales_df['Category'], values=clean_sales_df['Sales'], aggfunc='sum')

plt.figure(figsize=(10, 6))
sns.heatmap(city_category, annot=True, fmt=".0f", cmap="YlGnBu")
plt.title("Revenue Heatmap: City vs Category")
plt.tight_layout()
plt.show()

### 3.13 Correlation Matrix

In [ ]:
plt.figure(figsize=(8, 5))
numeric_cols = clean_sales_df[['Age', 'UnitPrice', 'Quantity', 'Discount', 'Sales']]
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

## 4. Executive Business KPIs & Category Performance

In [ ]:
net_revenue = clean_sales_df['Sales'].sum()
total_orders = clean_sales_df['OrderID'].nunique()
total_customers = clean_sales_df['CustomerID'].nunique()
total_units_sold = clean_sales_df['Quantity'].sum()

aov = net_revenue / total_orders                            # Average Order Value
arpu = net_revenue / total_customers                        # Average Revenue Per User
avg_items_per_order = total_units_sold / total_orders        # Basket Size

total_discount_loss = clean_sales_df['Discount_Amount'].sum()
discount_loss_pct = (total_discount_loss / (net_revenue + total_discount_loss)) * 100

returned_orders = len(returns_df)
return_rate_pct = (returned_orders / len(df)) * 100

print("==========================================")
print("          EXECUTIVE BUSINESS KPIs         ")
print("==========================================")
print(f"• Net Revenue:                 ₹{net_revenue:,.2f}")
print(f"• Total Orders:                {total_orders:,}")
print(f"• Unique Customers:            {total_customers:,}")
print(f"• Average Order Value (AOV):   ₹{aov:.2f}")
print(f"• Revenue Per User (ARPU):     ₹{arpu:.2f}")
print(f"• Average Basket Size:         {avg_items_per_order:.2f} items")
print(f"• Return/Cancellation Rate:    {return_rate_pct:.2f}% ({returned_orders} orders)")
print(f"• Discount Revenue Leakage:    ₹{total_discount_loss:,.2f} ({discount_loss_pct:.2f}% of potential sales)")
print("==========================================\n")

# Category Breakdown KPI Table
category_kpi = clean_sales_df.groupby('Category').agg(
    Total_Revenue=('Sales', 'sum'),
    Total_Orders=('OrderID', 'count'),
    Avg_Order_Value=('Sales', 'mean'),
    Discount_Loss=('Discount_Amount', 'sum')
).reset_index().sort_values(by='Total_Revenue', ascending=False)

category_kpi['Revenue_Share_%'] = (category_kpi['Total_Revenue'] / net_revenue) * 100

print("==========================================")
print("         CATEGORY PERFORMANCE KPIs        ")
print("==========================================")
print(category_kpi.to_string(index=False))

## 5. Final Business Recommendations

In [ ]:
from IPython.display import display, Markdown

conclusion_text = """
# 📌 Final Business Recommendation & Strategy Summary

1. **Fix Discount Leakage (Save ₹282.15K):** Stop offering flat discounts up to 30%. Switch to threshold-based discounting (e.g., *Spend ₹150 to get 10% off*) to actually raise the basket size above 1.88 items.
2. **Re-engineer VIP Loyalty Program:** VIP Average Order Value (₹68.63) is slightly below Regular buyers (₹70.20). Introduce exclusive premium tech bundles to motivate higher order value.
3. **Double Down on Electronics:** Electronics drives **50.83% of total revenue**. Ensure supply chain and inventory stocking priority is given to high-demand electronic items across all major cities.
"""

display(Markdown(conclusion_text))